### Reading Data from Files (csv & json)



In [0]:
df = spark.read.format('csv').option('inferSchema', True).option('header',True).load('/Volumes/workspace/default/tutorial/BigMart Sales.csv')
df.display()

In [0]:
df_jason = spark.read.format('json').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/tutorial/drivers.json')
df_jason.display()

###Print Schema Defination

In [0]:
df.printSchema()

### Customize Schema : manual ddl & function structType

In [0]:
my_ddl_schema = '''
                Item_Identifier string,
                Item_Weight string,
                Item_Fat_Content string,
                Item_Visibility double, 
                Item_Type string,
                Item_MRP double,
                Outlet_Identifier string,
                Outlet_Establishment_Year integer,
                Outlet_Size string,
                Outlet_Location_Type string,
                Outlet_Type string,
                Item_Outlet_Sales double
'''
df = spark.read.format('csv')\
                .schema(my_ddl_schema)\
                .option('header',True)\
                .load('/Volumes/workspace/default/tutorial/BigMart Sales.csv')
df.display()
df.printSchema()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

my_strct_schema = StructType([
                            StructField('Item_Identifier', StringType(), True),
                            StructField('Item_Weight', StringType(), True),
                            StructField('Item_Fat_Content', StringType(), True),
                            StructField('Item_Visibility', StringType(), True),
                            StructField('Item_Type', StringType(), True),
                            StructField('Item_MRP', StringType(), True),
                            StructField('Outlet_Identifier', StringType(), True),
                            StructField('Outlet_Establishment_Year', StringType(), True),
                            StructField('Outlet_Size', StringType(), True),
                            StructField('Outlet_Location_Type', StringType(), True),
                            StructField('Outlet_Type', StringType(), True),
                            StructField('Item_Outlet_Sales', StringType(), True)
                            ])
df = spark.read.format('csv')\
                .schema(my_strct_schema)\
                .option('header',True)\
                .load('/Volumes/workspace/default/tutorial/BigMart Sales.csv')
df.printSchema()
df = spark.read.format('csv')\
                .schema(my_ddl_schema)\
                .option('header',True)\
                .load('/Volumes/workspace/default/tutorial/BigMart Sales.csv')


### SELECT columns

In [0]:
df = spark.read.format('csv')\
                .option('inferSchema', True)\
                .option('header',True)\
                .load('/Volumes/workspace/default/tutorial/BigMart Sales.csv')
df_sel = df.select('Item_Identifier','Item_Weight','Item_Fat_Content')
df_sel.display()
df_col = df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content'))
df_col.display()


###ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()


### FILTER Data

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

In [0]:
df.filter((col('Item_Type')=='Soft Drinks')&(col('Item_Weight')<10)).display()

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

### WITHCOLUMN function - imp col functions: new, modify, replace, typecast, sort etc


In [0]:
df_new_col = df.withColumn('flag',lit("NEW"))
df_new_col.display()

In [0]:
df_mod_col = df.withColumn('multiply_col', col('Item_Weight')*col('Item_MRP'))
df_mod_col.display()

In [0]:
df_rpl_col = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","REG",))\
                .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","LF",))
df_rpl_col.display()

In [0]:
df_typ_cast = df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))
df_typ_cast.printSchema()

### SORT / ORDER BY

In [0]:
df_sort_scn1 = df.sort(col('Item_Weight').desc())
df_sort_scn1.display()

In [0]:
df_sort_scn2 = df.sort(col('Item_Visibility').asc())
df_sort_scn2.display()


In [0]:
df_sort_scn3 = df.sort(['Item_Weight','Item_Visibility'],ascending=[0,0])
df_sort_scn3.display()

In [0]:
df_sort_scn4 = df.sort(['Item_Weight','Item_Visibility'],ascending=[0,1])
df_sort_scn4.display()

### Limit & Drop

In [0]:
df_lmt_col = df.limit(10)
df_lmt_col.display()

In [0]:
df_drp_scn1 = df.drop('Item_Visibility')
df_drp_scn1.display()

In [0]:
df_drp_scn2 = df.dropDuplicates()
df_drp_scn2.display()

In [0]:
df_drp_scn3 = df.dropDuplicates(subset=['Item_Type'])
df_drp_scn3.display()

In [0]:
df_dist_col = df.distinct()
df_dist_col.display()


### UNION Function

In [0]:
data1 = [('1','Ked'),
         ('2','Sid')]
schema1 = 'id STRING, name STRING'
df1 = spark.createDataFrame(data = data1, schema = schema1)

data2 = [('3','rahul'),
         ('4','jas')]
schema2 = 'id STRING, name STRING'
df2 = spark.createDataFrame(data = data2, schema = schema2)

df1.display()
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

### String Functions

In [0]:
df_init_col = df.select(initcap('Item_Type'))
df_init_col.display()
df_lwr_col = df.select(lower('Item_Type'))
df_lwr_col.display()
df_upr_col = df.select(upper('Item_Type'))
df_upr_col.display()

### DATE Functions: Current_date, Date_add, Date_sub

In [0]:
df_curdt_col = df.withColumn('Insert_dt', current_date())
df_curdt_col.display()
df_dtadd_col = df_curdt_col.withColumn('Week_after',date_add('Insert_dt',7))
df_dtadd_col.display()
df_dtsub_col = df_curdt_col.withColumn('Week_before',date_sub('Insert_dt',7))
df_dtsub_col.display()

In [0]:
df_dtdif_col = df_dtadd_col.withColumn('Duration',datediff('Week_after','Insert_dt'))
df_dtdif_col.display()

In [0]:
df_dtform_col = df_curdt_col.withColumn('Insert_dt',date_format('Insert_dt','MM/dd/yyyy'))
df_dtform_col.display()

### NULL Handling